<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Rules and colour histograms

**[Rules and colour histograms](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/)** · Vision AI: See, Count and Inspect · Module 2, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** sort product photos with a hand-written rule and with a colour histogram and nearest neighbours, add both to the comparison table, and see where they break.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Pixels and colour spaces, from [Images are numbers](https://learning.nextia-ai.com/courses/vision/m01/images-are-numbers/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M02-L01-rules-and-colour-histograms/rules-and-colour-histograms-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "449365badb665189b8bdc93d2ebb6f8d861d8ef1627d1c1a548e996960b89bda",
    "archives/extras.zip": "c2d0ecd8e6fda3846550f27aa054877a0f9daa4f6934d315907ecf107427a104",
    "compare.py": "95b41e2ed829d1e1be658947c266f53d571d9b441c4ed228d2cdec2750f4ed50",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The photos and their splits

In [ ]:
catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
studio = pd.read_csv("data/catalogue/new_studio_manifest.csv")
print(len(train), "train,", len(test), "test,", len(studio), "new studio photos")

## 2. A rule that a person writes

Priya looked at 20 photos and wrote four rules: a bright centre on a dark border is a candle; many green pixels is a plant pot; a very light photo is a cushion; everything else is a chair (the largest class).

> **Predict.** What accuracy does this rule reach on the 160 test photos: about 0.25, 0.5 or 0.75?

In [ ]:
def rule(path):
    a = np.asarray(Image.open(path).convert("RGB").resize((64, 64)), float)
    hsv = np.asarray(Image.open(path).convert("RGB").resize((64, 64)).convert("HSV"), float)
    centre = a[24:40, 24:40].mean()
    border = np.concatenate([a[:8].reshape(-1, 3), a[-8:].reshape(-1, 3)]).mean()
    green = ((hsv[..., 0] > 50) & (hsv[..., 0] < 120) & (hsv[..., 1] > 80)).mean()
    if border < 60 and centre > 120:
        return "candle"
    if green > 0.12:
        return "plant_pot"
    if a.mean() > 170:
        return "cushion"
    return "chair"

pred = [rule("data/" + f) for f in test["file"]]
s = cp.classification_scores(list(test["label"]), pred, CLASSES)
print("accuracy", s["accuracy"], "| worst class", s["worst_class"], s["worst_recall"])
print({k: v["recall"] for k, v in s["per_class"].items()})

> **Check.** The accuracy is 0.269. The rule finds candles and plant pots, and never says bowl, lamp, mug or vase.

## 3. A colour histogram

A colour histogram counts how many pixels have each colour. We use the HSV colour space (hue, saturation, value) with 8 × 4 × 4 = 128 bins. Two photos with similar colours get similar histograms.

In [ ]:
def colour_histogram(path, bins=(8, 4, 4)):
    """Hue, saturation and value in 8 x 4 x 4 bins; the square root makes distances compare histograms fairly."""
    hsv = np.asarray(Image.open(path).convert("RGB").resize((128, 128)).convert("HSV")).reshape(-1, 3)
    h, _ = np.histogramdd(hsv, bins=bins, range=((0, 256), (0, 256), (0, 256)))
    return np.sqrt(h.ravel() / h.sum())

h = colour_histogram("data/" + train["file"][0])
print(h.shape, round(float((h ** 2).sum()), 3))
plt.figure(figsize=(8, 2)); plt.bar(range(128), h); plt.title("the colour histogram of one photo"); plt.show()

## 4. Nearest neighbours

For each test photo, find the 5 training photos with the closest histograms, and take the most common label among them.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
Xtr = np.stack([colour_histogram("data/" + f) for f in train["file"]])
Xte = np.stack([colour_histogram("data/" + f) for f in test["file"]])
Xst = np.stack([colour_histogram("data/" + f) for f in studio["file"]])
knn = KNeighborsClassifier(5, n_jobs=1).fit(Xtr, train["label"])
for name, X, y in (("catalogue test", Xte, test["label"]), ("new studio", Xst, studio["label"])):
    s = cp.classification_scores(list(y), list(knn.predict(X)), CLASSES)
    print(f"{name:15s} accuracy {s['accuracy']}  worst {s['worst_class']} {s['worst_recall']}")

> **Check.** You should see 0.263 on the test photos and 0.225 on the new studio's photos: the same products under warmer light lose about 4 points. The histogram knows colour and nothing else.

Now compare with the course's table: the pretrained models are near 0.9.

In [ ]:
table = cp.Table.load("data/comparison_table.json")
table.show("catalogue", columns=("method", "setting", "value", "worst", "worst_value", "ms_per_image", "labelled_images"))

## 5. Your turn: how many neighbours?

> **Your turn.** Train the nearest-neighbour classifier with `k = 1` and with `k = 15`, and put their test accuracies in `acc_k1` and `acc_k15` (numbers with 3 decimals, as `cp.classification_scores` gives them).

In [ ]:
acc_k1 = cp.classification_scores(list(test["label"]), list(KNeighborsClassifier(1, n_jobs=1).fit(Xtr, train["label"]).predict(Xte)))["accuracy"]
acc_k15 = cp.classification_scores(list(test["label"]), list(KNeighborsClassifier(15, n_jobs=1).fit(Xtr, train["label"]).predict(Xte)))["accuracy"]
print(acc_k1, acc_k15)

In [ ]:
expect('acc_k1', acc_k1, 0.275, tolerance=0.002)

In [ ]:
expect('acc_k15', acc_k15, 0.325, tolerance=0.002)

## 6. When does a colour rule win?

A colour histogram is fast (under 1 ms a photo) and needs no model. It works when the camera and the light are fixed and the classes differ in colour, as on a factory line with one product of each colour. Larkfield's catalogue photos come from many cameras and rooms: there, it loses badly. Write one sentence for Kwame: where in the warehouse would a colour rule be enough?

## Next

You built the first two rows of the comparison table. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Hand-made features](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.